## Kinetics400 Video Retreival and Concept Logic with VideoMAE

This notebook mimics [example_search_concept_logic](example_search_concept_logic.ipynb), but changing the modality to video.

In order to adapt Text-To-Concept to video, the CLIP text and image representations are substituted by using the model ViCLIP. Moreover, VideoMAE is used as a video feature extractor is used.

The tasks in this notebook are:
+ Video Retrieval: finding videos which have some concepts (e.g., "a video in nature").
+ Concept Logic:

### Preliminaries
In this section, we import the required libraries and initialize standard transformations necessary for loading datasets. It is worth mentioning that certain models require input normalization, while others do not.

In [1]:
%load_ext autoreload
%autoreload 2
import torch
import torchvision
import numpy as np
from tqdm import tqdm
from TextToConcept import TextToConcept
import my_utils

/Users/rodrigopaganini/miniconda3/envs/mva-t2c/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/rodrigopaganini/miniconda3/envs/mva-t2c/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/Users/rodrigopaganini/miniconda3/envs/mva-t2c/lib/python3.10/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/Users/rodrigopaganini/master/xai/project/Text-to-concept/ViCLIP/viclip_text.py:4: UserWarning: pkg_res

In [2]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [3]:
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

Using device: mps


In [4]:
# std_transform_without_normalization = torchvision.transforms.Compose([
#     torchvision.transforms.Resize(224),
#     torchvision.transforms.CenterCrop(224),
#     torchvision.transforms.ToTensor()])


# std_transform_with_normalization = torchvision.transforms.Compose([
#     torchvision.transforms.Resize(224),
#     torchvision.transforms.CenterCrop(224),
#     torchvision.transforms.ToTensor(), 
#     torchvision.transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])



### VideoMAEv2
In this part, we load VideoMAE model.

As in the original image implementation in order to use ``TextToConcept`` framework, model should implement these functions/attributes:
+ ``forward_features(x)`` that takes a tensor as the input and outputs the representation (features) of input $x$ when it is passed through the model.
+ ``get_normalizer`` should be the normalizer that the models uses to preprocess the input. e.g., Resnet18, uses standard ImageNet normalizer.
+ Attribute ``has_normalizer`` should be `True` when normalizer is need for the model.

In video processing, this is achieved by means of the `VideoMAETTCTWrapper`.

In [5]:
from transformers import VideoMAEModel
from video_utils import VideoMAETTCTWrapper

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# feature_extractor = VideoMAEVideoProcessor.from_pretrained("MCG-NJU/videomae-small-finetuned-ssv2")  # TODO unnecessary??
videomae_model = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base")
videomae_model = videomae_model.to(device)

model = VideoMAETTCTWrapper(videomae_model, normalizer=torchvision.transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD))

Loading weights: 100%|██████████| 184/184 [00:00<00:00, 6906.61it/s]
VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias      | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED |  | 
decoder.norm.bias                                                    | UNEXPECTED |  | 
mask_token                                                           | UNEXPECTED |  | 
decoder.head.weight                                                  | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.q_bias       | UNEXPECTED |  

### Linear Aligner

<b>Initiating Text-To-Concept Object</b><br>
In this section, we initiate ``TextToConcept`` object which turns the vision encoder (e.g., VideoMAE) into a model capable of integrating language and vision. By doing so, we enable the utilization of certain abilities present in vision-language models.

In [6]:
text_to_concept = TextToConcept(model, model_name='videomae', input_type='video')

<b>Loading the Linear Aligner</b><br>
We can also use an already existing linear aligner, to do so, we use the function ``load_linear_aligner``.

In [7]:
path = 'pretrained_aligners/videomae_base_aligner_k400_v2.pth'
text_to_concept.load_linear_aligner(path)

In [8]:
dset_name = 'k400_val'

<b>Image Retrieval (1)</b><br>
In this section, we search for prompt 
`"polka-dotted"`.

In [9]:
templates = ['itap of a {}', 'a bad photo of the {}', 'a origami {}', 'a photo of the large {}',
             'a {} in a video game', 'art of the {}', 'a photo of the small {}']

In [97]:
sorted_inds, sims = text_to_concept.search(
    dset=dset,
    dset_name='imagenet_val', 
    prompts=[prompt.format('polka-dotted') for prompt in templates])

  0%|          | 0/2475 [00:00<?, ?it/s]/Users/rodrigopaganini/miniconda3/envs/mva-t2c/lib/python3.10/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 0/2475 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
reps, labels = text_to_concept.get_dataset_reps(dset, 'imagenet_val', True) #we need labels for visualization.

<b>Visualization (I)</b><br>
Visualization of top images:

function `visualize_top_images` takes these inputs:
+ `indices`: sorted indices of the images, based on similarity to concept.
+ `dset`: dataset
+ `num_of_image`: number of images to visualize,
+ `num_of_images_in_each_row`: number of images in each row.

This function plots top images based on similarity to concept.

In [ ]:
my_utils.visualize_top_images(sorted_inds, dset, 16, 4)

<b>Visualization (II)</b><br>
Visualization of top images in class-wise manner:

function `visualize_classwise_top_bot_images` takes these inputs:
+ `concept_str`: concept to which we consider similarity, 
+ `sims`: similariy score of images in the dataset to the concept,
+ `labels`: label of the images in the dataset, 
+ `dset`: dataset, 
+ `list_of_classes`: list of class names in the dataset

This function finds top-4 classes based on the similarity to the concept and plots most and least similar images within each of those classes.

In [ ]:
my_utils.visualize_classwise_top_bot_images('polka-dotted', sims, labels, dset, my_utils.imagenet_classes)

In [ ]:
my_utils.visualize_classwise_top_bot_images('polka-dotted', sims, labels, dset, my_utils.imagenet_classes)

<b>Image Retrieval (2)</b><br>
In this section, we search for prompt 
`"in a tree"`.
In order to obtain an encoding for this concept, we average the encoding of all these prompts:
```
    [template.format(c) + 'in a tree' for c in imagenet_classes for template in templates]

```

Function `encode_concepts_by_class` takes a list of concepts and returns the encoding for all those concepts in a 2D tensor (e.g. with the following piece of code, we obtain encoding for concepts `in a tree` and `in a road`).

In [ ]:
tensors = my_utils.encode_concepts_by_class(['in a tree', 'in a road'])

By having access to encoded concepts, we can call `search_with_encoded_concepts` function which is pretty similar to `search` except it takes tenor instead of prompts.

In [ ]:
sorted_inds, sims = text_to_concept.search_with_encoded_concepts(
    dset=dset,
    dset_name='imagenet_val', 
    vec=tensors[0].unsqueeze(0).to('mps'),)

In [ ]:
my_utils.visualize_top_images(sorted_inds, dset, 16, 4)

In [ ]:
my_utils.visualize_classwise_top_bot_images('in a tree', sims, labels, dset, my_utils.imagenet_classes)

<b>Concept Logic (1)</b><br>
In this section, we search for some particular images using ConceptLogic.

For the first example, we look for a photos of `dog`, in `the beach` while `sunset`.

In [ ]:
indices, _ = text_to_concept.concept_logic(
    dset=dset,
    dset_name='imagenet_val', 
    list_of_prompts=[[template.format(c) for template in templates] for c in ['a dog', 'the beach', 'the sunset']],
    signs=[1, 1, 1],
    scales=[1.8, 1.5, 1.5])

print(f'number of retrieved images: {len(indices)}')

<b>Visualization</b><br>
Visualization of top images:

????

In [ ]:
my_utils.visualize_top_images(indices, dset, 3, 3)

<b>Concept Logic (2)</b><br>
Another example is photos of `orange cat` not `indoors`.


In [ ]:
indices, _ = text_to_concept.concept_logic(
    dset=dset,
    dset_name='imagenet_val',
    list_of_prompts=[[template.format(c) for template in templates] for c in ['a cat', 'orange', 'indoors']],
    signs=[1, 1, -1],
    scales=[2.5, 1.5, 0])

print(f'number of retrieved images: {len(indices)}')

<b>Visualization</b><br>
Visualization of top images:

In [ ]:
my_utils.visualize_top_images(indices, dset, 5, 5)

## Evaluation over K400

In [10]:
from pathlib import Path
import json

from video_utils import load_k400_split, VideoMAETTCTWrapper, CTHWToTCHW, DivideBy255, \
    ToTensorTuple, SizedLabeledVideoDataset
from pytorchvideo.transforms import UniformTemporalSubsample, ApplyTransformToKey
from torchvision.transforms import Compose, Resize, CenterCrop
from pytorchvideo.data import UniformClipSampler


preprocessing_without_normalization = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(16),
            DivideBy255(),
            Resize((224, 224)),
            CenterCrop(224),
            CTHWToTCHW(),
            # Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]),
    ),
    ToTensorTuple(['video', 'label', 'video_name']),
])

# SSV2_ROOT = Path("../dataset/ssv2")
# LABELS_DIR = SSV2_ROOT / "labels"  # contains train.json, validation.json, test.json, labels
K400_ROOT = Path("dataset/k400")
K400_CLASSES_PATH = Path("dataset/k400/kinetics_classnames.json")

with open(K400_CLASSES_PATH, 'r') as f:
    class_to_idx = json.load(f)
class_to_idx = {k.strip('"'): int(v) for k, v in class_to_idx.items()}

labeled_video_paths = load_k400_split(
    "test",
    K400_ROOT / "test/1/videos_val",
    labels_path= K400_ROOT / "test/1/kinetics400_val_list_videos.txt",
)
clip_sampler = UniformClipSampler(clip_duration=10.0)

# indices = np.random.choice(len(labeled_video_paths), size=SUBSET_NUM_SAMPLES, replace=False)
# subset_paths = [labeled_video_paths[i] for i in indices]

dset = SizedLabeledVideoDataset(
    video_sampler=torch.utils.data.SequentialSampler,
    # labeled_video_paths=subset_paths,
    labeled_video_paths=labeled_video_paths,
    clip_sampler=clip_sampler,
    transform=preprocessing_without_normalization,
)
print(len(dset))

19796


**Video Retrieval**

**Concept Logic**

In [11]:
# import os
# import numpy as np

# labels = np.array([info["label"] for _, info in dset._labeled_videos], dtype=np.int64)

# os.makedirs("datasets/videomae/k400_val", exist_ok=True)
# np.save("datasets/videomae/k400_val/vision_model_labels.npy", labels)

# print(labels.shape)
# print("saved to datasets/videomae/k400_val/vision_model_labels.npy")

In [12]:
text_to_concept.saved_dsets['k400_val'] = (
    'data/reps_videomae_k400_val/vision_model_reps_k400_val.npy', 
    'data/reps_videomae_k400_val/vision_model_reps_labels_k400_val.npy',
    'data/reps_videomae_k400_val/vision_model_reps_names_k400_val.npy'
)

In [13]:
import shutil

concept = "danger"
templates = ['a video of a person with {}', 'a tutorial of {}', 'a clip of {}', 'a bad video of the {}', 'a video of someone {}',]

indices, sims = text_to_concept.search(
    dset=dset,
    dset_name='k400_val', 
    prompts=[prompt.format(concept) for prompt in templates],
)
names = np.load(text_to_concept.saved_dsets['k400_val'][2], allow_pickle=True)

Path(f"data/to_view/search_videos--{concept}").mkdir(parents=True, exist_ok=True)
for i,name in enumerate(names[indices][:10]):
    fpath = K400_ROOT / f"test/1/videos_val/{name}.mp4"
    if fpath.exists():
        print(f"copying {fpath.name} with similarity {sims[i]:.4f}")
        shutil.copy(fpath, f"data/to_view/search_videos--{concept}/")
    else:
        print("discarding element", i)

copying qCocNTBkeWY.mp4 with similarity 0.2046
copying L1mDlSDbb9k.mp4 with similarity 0.1858
copying 7Jn8PS8kiLo.mp4 with similarity 0.1917
copying 3rzLqKZCLNw.mp4 with similarity 0.1875
copying XilAaJ_r4tA.mp4 with similarity 0.1555
copying WnRTUcOFc2U.mp4 with similarity 0.1273
copying DkSFdTiSDa8.mp4 with similarity 0.1785
copying e4YEsKp3pJ8.mp4 with similarity 0.1853
copying UOvtaFUITEw.mp4 with similarity 0.1732
copying Z0Gm7WTyk-o.mp4 with similarity 0.1480


In [ ]:
indices, sims = text_to_concept.concept_logic(
    dset=dset,
    dset_name='k400_val', 
    list_of_prompts=[prompt_list for prompt_list in concepts_and_prompts.values()],
    # list_of_prompts=[[concept.format(c) for template in templates] for c in concepts],
    signs=signs,
    scales=scales
)
names = np.load(text_to_concept.saved_dsets['k400_val'][2], allow_pickle=True)

Path("data/to_view/retrieved_videos").mkdir(parents=True, exist_ok=True)
for name in names[indices][:10]:
    fpath = K400_ROOT / f"test/1/videos_val/{name}.mp4"
    if fpath.exists():
        shutil.copy(fpath, "data/to_view/retrieved_videos/")

Note: It's important to tune the signs & scales to have only a few examples selected. Otherwise, top scores are misleading

In [ ]:
indices, _ = text_to_concept.concept_logic(
    dset=dset,
    dset_name='imagenet_val', 
    list_of_prompts=[[template.format(c) for template in templates] for c in ['frog', 'green']],
    signs=[1, 1],
    scales=[2.5, 1.5])

print(f'number of retrieved images: {len(indices)}')
my_utils.visualize_top_images(indices, dset, 3, 3)